## Obtaining, parsing and understanding ALT loci

In [1]:
import os
import gzip
import tempfile
import logging
import collections
import urllib.request
from collections import defaultdict
from ftplib import FTP

In [2]:
logging.basicConfig(level=logging.DEBUG)

In [3]:
def nav_to_root():
    ftp = FTP('ftp.ncbi.nlm.nih.gov')
    ftp.login()
    dr = 'genomes/all/GCA/000/001/405/GCA_000001405.28_GRCh38.p13/GCA_000001405.28_GRCh38.p13_assembly_structure'
    ftp.cwd(dr)
    return ftp

In [4]:
alt_grps = []

def handle_line(st):
    fn = st.split()[-1]
    if fn.startswith('ALT_'):
        alt_grps.append(fn)

nav_to_root().retrlines('LIST', handle_line)

'226 Transfer complete'

Names like `ALT_REF_LOCI_1` are "alternate locus groups"

In [5]:
alt_grps

['ALT_REF_LOCI_1',
 'ALT_REF_LOCI_10',
 'ALT_REF_LOCI_11',
 'ALT_REF_LOCI_12',
 'ALT_REF_LOCI_13',
 'ALT_REF_LOCI_14',
 'ALT_REF_LOCI_15',
 'ALT_REF_LOCI_16',
 'ALT_REF_LOCI_17',
 'ALT_REF_LOCI_18',
 'ALT_REF_LOCI_19',
 'ALT_REF_LOCI_2',
 'ALT_REF_LOCI_20',
 'ALT_REF_LOCI_21',
 'ALT_REF_LOCI_22',
 'ALT_REF_LOCI_23',
 'ALT_REF_LOCI_24',
 'ALT_REF_LOCI_25',
 'ALT_REF_LOCI_26',
 'ALT_REF_LOCI_27',
 'ALT_REF_LOCI_28',
 'ALT_REF_LOCI_29',
 'ALT_REF_LOCI_3',
 'ALT_REF_LOCI_30',
 'ALT_REF_LOCI_31',
 'ALT_REF_LOCI_32',
 'ALT_REF_LOCI_33',
 'ALT_REF_LOCI_34',
 'ALT_REF_LOCI_35',
 'ALT_REF_LOCI_4',
 'ALT_REF_LOCI_5',
 'ALT_REF_LOCI_6',
 'ALT_REF_LOCI_7',
 'ALT_REF_LOCI_8',
 'ALT_REF_LOCI_9']

### Get alignment GFFs

In [6]:
gffs, grp_gffs = [], []
ftp = nav_to_root()
for alt_grp in alt_grps:
    dr_alt = os.path.join(alt_grp, 'alt_scaffolds', 'alignments')
    ftp.cwd(dr_alt)

    def handle_alignment_line(st):
        fn = st.split()[-1]
        if fn.endswith('.gff'):
            grp_gffs.append((alt_grp, fn))
            gffs.append(fn)

    ftp.retrlines('LIST', handle_alignment_line)
    ftp.cwd('../../..')

In [7]:
print(len(gffs), len(set(gffs)))

261 261


Is this the expected number of ALT contigs?

In [8]:
gffs[0]

'GL000250.2_CM000668.2.gff'

In [9]:
srcs, dsts = [], []
for gff in gffs:
    toks = gff[:-4].split('_')
    srcs.append(toks[0])
    dsts.append(toks[1])

In [10]:
print(len(srcs), len(set(srcs)))

261 261


In [11]:
print(len(dsts), len(set(dsts)))

261 23


In [12]:
dst_counter = collections.Counter(dsts)

In [13]:
for k, v in dst_counter.items():
    print((k, v))

('CM000668.2', 16)
('CM000666.2', 11)
('CM000679.2', 18)
('CM000667.2', 13)
('CM000663.2', 12)
('CM000664.2', 15)
('CM000665.2', 16)
('CM000669.2', 9)
('CM000671.2', 5)
('CM000672.2', 4)
('CM000673.2', 12)
('CM000674.2', 13)
('CM000677.2', 9)
('CM000678.2', 6)
('CM000680.2', 10)
('CM000681.2', 43)
('CM000682.2', 4)
('CM000683.2', 7)
('CM000684.2', 9)
('CM000670.2', 16)
('CM000675.2', 6)
('CM000676.2', 4)
('CM000685.2', 3)


In [14]:
dst_counter.most_common(1)[0][0]

'CM000681.2'

In [15]:
# Parse attributes field of GFF line
def parse_attr(attr):
    targstr = ';Target='
    assert targstr in attr, attr
    toff = attr.index(targstr)
    target = attr[toff+len(targstr):attr.index(' ', toff)]
    gapstr = ';Gap='
    assert gapstr in attr, attr
    coff = attr.index(gapstr)
    cigar_st = attr[coff+len(gapstr):]
    cigar_ls = cigar_st.split(' ')
    ops = list(map(lambda x: x[0], cigar_ls))
    for op in ops:
        assert op in 'MDI'
    runs = list(map(lambda x: int(x[1:]), cigar_ls))
    return target, list(zip(ops, runs))

In [16]:
attr='ID=aln0;Target=GL000250.2 1 4672374 +;align_id=1411;batch_id=142943;bit_score=1.12029;common_component=0;e_value=4.83307e+11;expansion=1.96768;filter_score=7;merge_aligner=1;merge_options=58;num_ident=2348261;num_mismatch=6441;pct_coverage=50.3963;pct_identity_gap=33.7832;pct_identity_gapopen_only=99.688;pct_identity_ungap=99.7265;reciprocity=3;Gap=M44558 D8 M13552 D2 M3960 D4 M2109 D2 M33 I2 M25649 I6 M18550 D1 M68504 I1 M25289 I44253 D44253 M3450 D1 M345 I1 M3095 I39 M16670 I1 M48609 D2 M2804 I29384 D29384 M14754 I7 M3825 D2 M7033 D8 M3588 D1 M19578 D1 M1324 D2 M5435 I5 M2064 I2 M5713 D1 M3146 I1 M15 I1 M28 I1 M2026 I1 M14932 I1 M168 D1 M6579 D4 M683 I8 M47 I4 M1147 I87598 D87598 M5106 I2 M1514 D1 M3116 D1 M5039 D1 M8561 I19 M8035 I1 M2049 I1 M502 I2 M3290 I10 M6685 I1 M8656 I104192 D104192 M1686 D2 M5376 D1 M19253 I1 M7864 I1 M41 D6 M4451 I1 M27910 I1 M9946 D1 M17202 I6 M1218 I1 M3967 D2 M462 I1 M20759 I1 M3779 I1 M1268 I1 M1551 I2 M4147 I22 M1757 I1 M3444 I1 M6666 D4 M1349 D4 M1101 I2 M1503 I3 M1077 I1 M5745 I1 M5097 D1 M6153 D1 M4783 I6 M3323 D3 M965 I1 M1234 I28 M1194 I1 M2897 D1 M5932 I1 M8163 D10 M618 I5 M667 D8 M562 D1 M454 I1 M9434 I5 M6166 D1 M1064 D6 M1374 I2 M3816 I1 M1245 I2 M3424 D4 M722 D1 M502 D1 M3670 I1 M740 D1 M1693 D1 M6497 D2 M2700 I1 M199 D2 M141 D1 M1108 D2 M1448 I1 M9023 I4 M844 I19123 D19123 M145 I1 M11 I4 M297 D24 M337 I11 M813 I3 M530 I2 M1000 I1 M2520 I4 M576 I18 M209 I59377 D59240 M105 D32 M700 D3 M2315 D4 M807 I1 M291 I1 M3596 D7 M1953 D1 M2111 D1 M950 D1 M2209 I3 M858 I11 M2729 I1 M401 D1 M2337 I2 M7220 I1 M1025 I1 M3002 D3 M1282 D2 M671 D21 M709 I3 M936 D1 M320 D1 M100 I1 M72 I1 M602 D4 M179 D207 M684 D1 M1482 I1 M986 I18 M128 D4 M299 I1 M114 D5 M944 I14 M274 D3 M87 D1 M7749 I4 M881 D1 M477 D4 M730 I1 M599 D2 M2613 I1 M1624 D2 M4594 D8 M2534 I2 M1525 D1 M4266 I1 M108 D1 M2346 I14 M645 I15 M5578 D1 M1712 I4 M3267 D2 M2251 I1 M3322 D248 M2227 I2 M865 D2 M1154 I1 M63 I1 M300 I3 M691 I6 M919 D3 M948 D4 M1857 D12 M174 D9 M384 D1 M417 I8 M92 D2 M802 I2 M2665 D3 M1045 I3 M712 D2 M349 I3 M1412 D1 M1299 D5 M1483 I1 M4678 D1 M509 D2 M517 D1 M1011 D2 M970 D1 M160 I18 M577 D5668 M543 D2 M217 I8 M3 I1 M4 I6 M199 I1 M251 D1 M2595 I5 M381 I1 M829 D1 M350 D1 M281 D2 M314 D6 M1519 D1 M130 D1 M1276 D4 M1860 D3 M1040 D3 M103 D1 M24 D9 M168 I4 M1418 I5 M456 I1 M4 D1 M14 D1 M14 D4 M12 D1 M13 D1 M24 D3 M113 D1 M34 I3 M19 D79 M1507 D1 M1216 D2 M679 I2 M609 D1 M46 D1 M789 D2 M1538 D9 M417 I2 M180 I2 M195 D6 M106 I1 M2086 I2 M53 I91 M12 D2 M2065 D1 M2416 I1 M264 D1 M2251 D1 M2518 I3 M4215 D2 M74 D2 M154 I1 M383 I4 M372 I1 M2441 I4 M1890 D1 M123 I4 M441 I1 M470 D1 M66 D4 M103 D38 M1969 D2 M173 I1 M307 I1 M579 I2 M123 I2 M188 I1 M1214 D1 M1532 D1758 M538 I1 M23 I1 M152 D41 M196 I1 M92 I1 M1113 D144 M36 I1 M4347 I1 M168 I6 M1379 D3 M1685 I1 M5878 I4 M4169 D19 M1595 D1 M687 I4 M330 D1 M159 D22 M3841 I1 M5174 D3 M748 I24594 D24594 M686 I1 M11541 I1 M1843 D4 M8525 I19 M738 D1 M8058 I1 M5119 I1 M2913 D1 M374 I332 M6073 D1 M1967 I7 M3199 I1 M667 I1 M3869 D1 M36 I13 M1936 D4 M1137 I2 M211 I2 M345 I1 M1639 I4 M686 I14 M1032 D1 M9035 D4 M167 D2 M23964 D3 M1760 I3 M3716 I5 M643 I8 M2081 D4 M355 D1 M1829 I4 M2052 D12 M807 I9746 D9746 M338 D1 M1002 D1 M7912 I4 M4977 D1 M9275 I2934 D2931 M21530 D1 M4385 D7 M881 D8 M17594 D20 M2417 I1 M998 D10 M4239 I7 M4845 D2 M118 I5 M15722 D6 M12084 D3 M3052 D4 M2989 I1 M13524 D2 M3050 I104591 D33135 M2281 D7 M377 I3 M7256 I3 M419 I2 M10674 D17 M5362 I16440 D16440 M408 I1 M3061 I2 M5818 I1 M2424 I2 M2056 D1 M953 I4 M17262 I1 M9557 D2 M5358 D1 M14165 I2 M64144 I19 M12369 I49753 D49753 M608 D5 M2298 I3 M2327 D1 M414 I1 M61 D1 M235 D2 M2087 D1 M713 D8 M54 I3 M1027 I3 M2974 D1 M8189 D1 M140 I2 M700 D3 M150 D1 M742 D3 M3228 I2 M473 D2 M2032 I1 M3841 I1 M560 I1 M1108 I1 M8087 I1 M5394 I103872 D103871 M16267 I1 M18633 D1 M920 D1 M6820 I1 M1354 D1 M4363 D1 M12467 D1 M9375 D1 M323 I1 M1715 I154558 D154558 M4522 I85167 D85167 M6234 D3 M4009 D5 M6425 I2 M1102 I2 M33 I8 M2257 D1 M3217 I7 M2436 D14 M8066 I1 M2339 D1 M310 I1 M1583 D16 M343 I7 M155 I7 M3 I1 M61 D2 M62 I299 M920 D33 M1483 I1 M269 D3 M1609 I4 M171 D2 M609 I5 M429 I1 M1486 I9 M2736 I4 M1220 I25 M52 D1 M419 I2 M4854 I30 M793 I3 M9785 I2 M1144 I3687 D873 M205 I2 M1593 I3 M899 D89 M1660 D6 M2155 I1 M1164 I1 M1310 I1 M4637 I1 M21451 I401260 D401260 M2932 D2 M9966 D1 M6798 D19 M5789 I1 M21042 D1 M13887 I36782 D36752 M30443 I7 M150 I15 M2797 D2 M329 I10 M2475 I142825 D142825 M380 I4 M709 D1 M12092 D1 M2431 I1 M158 D1 M281 I1 M1573 D1 M11079 I1 M6938 D1 M14766 I1 M4187 I10 M11099 I3 M9487 D6 M467 I1 M2143 I1 M723 D6 M44 D1 M21 I1 M1733 D1 M259 I2 M295 I1 M128 D1 M3052 D1 M1419 I13507 D13507 M194 I1 M15841 D2 M3647 I1 M17240 I4 M3936 I3 M2976 I174972 D174973 M5315 D20 M2124 D1 M7445 I45267 D45267 M361 D2 M1466 I1 M6739 D1 M1422 D1 M324 I1 M1825 I1 M311 D1 M6568 I1 M5228 I1 M2254 I1 M6143 I1 M19952 D3 M12288 I1 M1884 I10 M512 I1 M2472 D1 M2859 I1 M2081 D2 M373 D4 M538 I103035 D103035 M3265 D21 M4471 D1 M3579 I6 M576 D2 M1120 I4 M168 D1 M2870 I21 M144 I1 M1440 D1 M625 D13 M144 D4 M1177 I2 M2392 D1 M3378 D1 M1906 D5 M344 I1 M1575 I1 M1256 I1 M2208 D7 M908 D17 M806 D323 M525 D1 M310 D2 M2978 D4 M3495 D4 M2869 I3653 D3651 M214 D2 M499 I1 M1405 I6 M1400 D1 M766 D1 M451 I1 M153 I1 M1000 I3 M702 I4 M481 I2 M538 I4 M219 D1 M742 D1 M541 I1 M88 I2 M704 I1 M330 D12 M432 I1 M4 I2 M324 D1 M3060 D2 M11853 D12 M839 D1 M14260 D2 M9661 I1 M3607 I39571 D39571 M1619 I2 M164 I7 M1617 I1 M1780 I2 M4539 I3 M3833 D1 M10010 I1 M3098 I3322 D3320 M1856 D1 M168 I1 M170 I4 M125 I7 M353 I1 M205 I6434 D31106 M119 D1 M40 D2 M158 D1 M46 D1 M51 D1 M103 I1 M193 D2 M71 D1 M355 D18 M40 I1 M445 D1 M348 D4 M97 D5 M291 I1 M119 I1 M24 D2 M41 D4 M87 D1 M123 I5 M233 I34 M8 I2 M86 I2 M81 D2 M77 D5 M89 D4 M289 I1 M4 I5 M97 I2 M28 I22 M462 I1 M81 D1 M55 I2 M136 D1 M16 I4 M135 I1 M189 D2 M24 D2 M149 D1 M39 D1 M53 I1 M62 D1 M157 I2 M30 D4 M41 D1 M105 I1 M444 I1 M14 D1 M145 I1 M130 D1 M267 D1 M32 D1 M107 D1 M585 I5 M21 I1 M79 I1 M453 D2 M54 D16 M17 I1 M47 D3 M108 D1 M6 D2 M196 D4 M119 I4 M103 D314 M225 I16 M40 I11 M16 I1 M252 I2 M4 D2 M101 I1 M40 D16 M354 D1 M7 D1 M8 I5 M10 I1 M45 I2 M3 I1 M65 I1 M59 D1 M144 I1 M105 I1 M115 D1 M120 D3 M71 I1 M22 D16 M106 D1 M22 I9 M6 I12 M412 D2 M100 D1 M57 D1 M154 I4 M53 I12 M340 I1 M145 I1 M364 D2 M240 I1 M360 D1 M172 I5 M231 I2 M82 I181838 D183306 M4833 D28 M133 I1 M776 I1 M491 I13 M499 I4 M20 I1 M336 I1 M8 D1 M686 D8 M110 I3 M44 I3 M30 D1 M47 I2 M195 I1 M62 I4 M40 I2 M33 I1 M19 I1 M434 I5 M39 I1 M94 D2 M68 I3 M796 I1 M552 D1 M304 I2 M21 I6 M453 I2 M192 D30 M94 I1 M77 I4 M178 D2 M43 D1 M540 D4 M938 I1 M1085 D1 M346 D1 M807 D2 M240 D9 M774 D1 M130 I1 M885 I2 M1003 D1 M376 I2 M252 D1 M1249 D1 M189 D1 M2307 D5 M318 D1 M1489 I1 M356 I1 M272 I323 M531 D2 M3937 I1 M2510 I1 M5051 D1 M4790 I1 M8240 I1 M706 I2 M6951 D2 M1868 D2 M826 I14 M38 I1 M1136 D1 M1767 D19 M345 D1 M2187 D1 M352 I1470 D1838 M326 D1 M23 I3 M743 D1 M797 D4 M47 D1 M48 D1 M46 D2 M1483 I1 M2148 I3 M163 I1 M1691 D1 M1588 D1 M145 I1 M323 I4 M1863 I1 M1371 D4 M145 I3 M2839 I3 M1345 D1 M23 I2 M1425 I1 M1773 D1 M392 I1 M240 D1 M238 D1 M180 I1 M843 D2 M85 D1 M10 I315 M321 I1 M1582 D2 M2010 I6 M574 I2 M480 I1 M154 I1 M157 I1 M26 D1066 M119 D1 M8640 D10 M2104 D1 M511 I2 M1704 I18 M2593 D177 M773 I10 M9789 D3 M939 I2 M10178 D2 M2337 I2 M6967 D1 M162 I1 M730 D3 M658 I1 M1570 D1 M736 I1 M6 I2 M1007 D1 M158 D10 M2096 D1 M1247 I1 M1055 I1 M491 I1 M1482 D1 M1947 D2 M891 I1 M169 I4 M1361 I1 M1127 D4 M155 I6 M192 D2 M1635 D13 M585 D16 M416 D2 M67 I3 M641 D1 M571 D1 M541 D2 M844 I5 M990 D1 M1303 D7 M1170 D1 M1236 D4 M2115 I5 M1093 D2 M9357 I8 M2112 D1 M4166 D1 M3476 I16 M7063 D1 M65 I3 M4293 D1 M3181 D4 M9494 I4 M11278 I2 M4856 I3 M6082 I1 M955 D1 M1720 I6 M17 I1 M8156 I53709 D53709 M818 I9 M3346 I14 M357 I3 M2454 I1 M1180 I4 M2148 D1 M20 I1 M474 D1 M1796 D4 M9749 D206 M9 D4 M20 I2 M13529 I1 M1837 I1 M2276 I1 M1824 I1 M1990 I1 M9965 D18 M2950 I6 M8470 D16 M18006 D1 M316 D26 M10857 D1 M7784 D1 M4078 D2 M5624 D1 M311 I15 M132 D19 M762 D1 M6 I12 M632 I1 M1572 D1 M215 I28 M28 D2 M217 I207856 D207856 M1739 I1 M702 D1 M5133 D2 M3096 I1 M315 I1 M490 D1 M2588 I3 M1013 D1 M3659 I1 M2413 D1 M2722 I4 M524'

In [17]:
parse_attr(attr)

('GL000250.2',
 [('M', 44558),
  ('D', 8),
  ('M', 13552),
  ('D', 2),
  ('M', 3960),
  ('D', 4),
  ('M', 2109),
  ('D', 2),
  ('M', 33),
  ('I', 2),
  ('M', 25649),
  ('I', 6),
  ('M', 18550),
  ('D', 1),
  ('M', 68504),
  ('I', 1),
  ('M', 25289),
  ('I', 44253),
  ('D', 44253),
  ('M', 3450),
  ('D', 1),
  ('M', 345),
  ('I', 1),
  ('M', 3095),
  ('I', 39),
  ('M', 16670),
  ('I', 1),
  ('M', 48609),
  ('D', 2),
  ('M', 2804),
  ('I', 29384),
  ('D', 29384),
  ('M', 14754),
  ('I', 7),
  ('M', 3825),
  ('D', 2),
  ('M', 7033),
  ('D', 8),
  ('M', 3588),
  ('D', 1),
  ('M', 19578),
  ('D', 1),
  ('M', 1324),
  ('D', 2),
  ('M', 5435),
  ('I', 5),
  ('M', 2064),
  ('I', 2),
  ('M', 5713),
  ('D', 1),
  ('M', 3146),
  ('I', 1),
  ('M', 15),
  ('I', 1),
  ('M', 28),
  ('I', 1),
  ('M', 2026),
  ('I', 1),
  ('M', 14932),
  ('I', 1),
  ('M', 168),
  ('D', 1),
  ('M', 6579),
  ('D', 4),
  ('M', 683),
  ('I', 8),
  ('M', 47),
  ('I', 4),
  ('M', 1147),
  ('I', 87598),
  ('D', 87598),
  ('M'

In [18]:
class Alignment(object):
    def __init__(self, seqname, target, st, en, cigar):
        self.seqname = seqname
        self.target = target
        self.st = int(st) - 1
        self.en = int(en)
        assert self.st < self.en
        self.cigar = cigar
    
    def __repr__(self):
        return '%s[%d:%d]->%s' % (self.seqname, self.st, self.en, self.target)

In [19]:
# Now let's actually parse the GFF alignments
tmp = tempfile.mkdtemp()
ftp = nav_to_root()
als = defaultdict(list)
for alt_grp, gff in grp_gffs:
    tmpfn = os.path.join(tmp, gff)
    ftp.cwd(os.path.join(alt_grp, 'alt_scaffolds', 'alignments'))
    with open(tmpfn, 'wb') as tmpfh:
        logging.info('Retrieving GFF "%s"...' % gff)
        ftp.retrbinary('RETR ' + gff, tmpfh.write, 1024)
    ftp.cwd('../../..')
    with open(tmpfn, 'rt') as fh:
        for ln in fh:
            if ln[0] == '#':
                continue
            toks = ln.rstrip().split('\t')
            assert 9 == len(toks), toks
            seqname, source, feature, st_str, en_str, score, strand, frame, attr = toks
            assert 'Genbank' == source
            assert 'match' == feature
            # Score is sometimes non-0; not sure what it's meant to convey
            #assert '0' == score
            assert '+' == strand
            assert '.' == frame
            if 'pct_identity_gap=100' in attr:
                logging.warning('SKIPPING alignment due to 100% identity')
                continue
            target, cigar = parse_attr(attr)
            als[seqname].append(Alignment(seqname, target, st_str, en_str, cigar))

INFO:root:Retrieving GFF "GL000250.2_CM000668.2.gff"...
INFO:root:Retrieving GFF "GL000257.2_CM000666.2.gff"...
INFO:root:Retrieving GFF "GL000258.2_CM000679.2.gff"...
INFO:root:Retrieving GFF "GL339449.2_CM000667.2.gff"...
INFO:root:Retrieving GFF "GL383518.1_CM000663.2.gff"...
INFO:root:Retrieving GFF "GL383519.1_CM000663.2.gff"...
INFO:root:Retrieving GFF "GL383520.2_CM000663.2.gff"...
INFO:root:Retrieving GFF "GL383521.1_CM000664.2.gff"...
INFO:root:Retrieving GFF "GL383522.1_CM000664.2.gff"...
INFO:root:Retrieving GFF "GL383526.1_CM000665.2.gff"...
INFO:root:Retrieving GFF "GL383527.1_CM000666.2.gff"...
INFO:root:Retrieving GFF "GL383528.1_CM000666.2.gff"...
INFO:root:Retrieving GFF "GL383530.1_CM000667.2.gff"...
INFO:root:Retrieving GFF "GL383531.1_CM000667.2.gff"...
INFO:root:Retrieving GFF "GL383532.1_CM000667.2.gff"...
INFO:root:Retrieving GFF "GL383533.1_CM000668.2.gff"...
INFO:root:Retrieving GFF "GL383534.2_CM000669.2.gff"...
INFO:root:Retrieving GFF "GL383539.1_CM000671.2.

INFO:root:Retrieving GFF "KI270840.1_CM000675.2.gff"...
INFO:root:Retrieving GFF "KI270841.1_CM000675.2.gff"...
INFO:root:Retrieving GFF "KI270842.1_CM000675.2.gff"...
INFO:root:Retrieving GFF "KI270843.1_CM000675.2.gff"...
INFO:root:Retrieving GFF "KI270844.1_CM000676.2.gff"...
INFO:root:Retrieving GFF "KI270845.1_CM000676.2.gff"...
INFO:root:Retrieving GFF "KI270846.1_CM000676.2.gff"...
INFO:root:Retrieving GFF "KI270847.1_CM000676.2.gff"...
INFO:root:Retrieving GFF "KI270848.1_CM000677.2.gff"...
INFO:root:Retrieving GFF "KI270849.1_CM000677.2.gff"...
INFO:root:Retrieving GFF "KI270850.1_CM000677.2.gff"...
INFO:root:Retrieving GFF "KI270851.1_CM000677.2.gff"...
INFO:root:Retrieving GFF "KI270852.1_CM000677.2.gff"...
INFO:root:Retrieving GFF "KI270853.1_CM000678.2.gff"...
INFO:root:Retrieving GFF "KI270854.1_CM000678.2.gff"...
INFO:root:Retrieving GFF "KI270855.1_CM000678.2.gff"...
INFO:root:Retrieving GFF "KI270856.1_CM000678.2.gff"...
INFO:root:Retrieving GFF "KI270857.1_CM000679.2.

In [20]:
als

defaultdict(list,
            {'CM000668.2': [CM000668.2[28734407:33367716]->GL000250.2,
              CM000668.2[79350007:79446911]->GL383533.1,
              CM000668.2[168796075:168951646]->KB021644.2,
              CM000668.2[169724798:169919706]->KI270797.1,
              CM000668.2[170272762:170535221]->KI270798.1,
              CM000668.2[68262878:68402208]->KI270799.1,
              CM000668.2[65774813:65944033]->KI270800.1,
              CM000668.2[127654802:128521146]->KI270801.1,
              CM000668.2[166220579:166285980]->KI270802.1,
              CM000668.2[28510119:33383765]->GL000251.2,
              CM000668.2[28734407:33361299]->GL000252.2,
              CM000668.2[28734407:33258200]->GL000253.2,
              CM000668.2[28734407:33391865]->GL000254.2,
              CM000668.2[28734407:33411973]->GL000255.2,
              CM000668.2[28691465:33480577]->GL000256.2,
              CM000668.2[32385053:32454732]->KI270758.1],
             'CM000666.2': [CM000666.2[683063

In [21]:
def sizes_from_cigar(cigar):
    ilen, totlen, dlen = 0, 0, 0
    for op, run in cigar:
        if op == 'M' or op == 'I':
            ilen += run
        if op == 'M' or op == 'D':
            dlen += run
        totlen += run
    return ilen, totlen, dlen

# Prints span of I+M ops, then total span (I+D+M), then D+M
sizes_from_cigar(als['CM000676.2'][0].cigar)

(322166, 322198, 318507)

In [22]:
# Reference span according to GFF record
als['CM000676.2'][0].en - als['CM000676.2'][0].st + 1

318508

The length of the reference substring involved in the alignment equals the span of the `M` and `D` CIGAR operations.  We can conclude that `D` corresponds to a column where a reference character appears opposite a gap.

### Get ALT sequences

In [23]:
def parse_fasta(fh):
    fa = {}
    current_short_name = None
    # Part 1: compile list of lines per sequence
    for ln in fh:
        try:
            ln = ln.decode()
        except AttributeError:
            pass
        if ln[0] == '>':
            # new name line; remember current sequence's short name
            long_name = ln[1:].rstrip()
            current_short_name = long_name.split()[0]
            fa[current_short_name] = []
        else:
            # append nucleotides to current sequence
            fa[current_short_name].append(ln.rstrip())
    # Part 2: join lists into strings
    for short_name, nuc_list in fa.items():
        # join this sequence's lines into one long string
        fa[short_name] = ''.join(nuc_list)
    return fa

In [24]:
ftp = nav_to_root()
fastas = []
for alt_grp in alt_grps:
    dr_alt = os.path.join(alt_grp, 'alt_scaffolds', 'FASTA')
    ftp.cwd(dr_alt)
    tmp = tempfile.mkdtemp()
    fns = []

    def handle_fasta_line(st):
        fn = st.split()[-1]
        if fn == 'alt.scaf.fna.gz':
            fns.append(fn)  # pretty simple
    
    def download_fasta(fn):
        tmpfn = os.path.join(tmp, fn)
        with open(tmpfn, 'wb') as tmpfh:
            logging.info('Retrieving "%s" for group "%s"...' % (fn, alt_grp))
            ftp.retrbinary('RETR ' + fn, tmpfh.write, 1024)
            logging.info('...Done')
        assert os.path.exists(tmpfn)
        with gzip.open(tmpfn, 'rt') as fh:
            return alt_grp, parse_fasta(fh)

    ftp.retrlines('LIST', handle_fasta_line)
    for fn in fns:
        fastas.append(download_fasta(fn))
    ftp.cwd('../../..')

combined = {}
tot_len = 0
for alt_grp, fasta in fastas:
    print('%s: %d' % (alt_grp, len(fasta)))
    tot_len += len(fasta)
    combined.update(fasta)

print('Combined len = %d' % len(combined))
print('Tot of each individual = %d' % tot_len)

INFO:root:Retrieving "alt.scaf.fna.gz" for group "ALT_REF_LOCI_1"...
INFO:root:...Done
INFO:root:Retrieving "alt.scaf.fna.gz" for group "ALT_REF_LOCI_10"...
INFO:root:...Done
INFO:root:Retrieving "alt.scaf.fna.gz" for group "ALT_REF_LOCI_11"...
INFO:root:...Done
INFO:root:Retrieving "alt.scaf.fna.gz" for group "ALT_REF_LOCI_12"...
INFO:root:...Done
INFO:root:Retrieving "alt.scaf.fna.gz" for group "ALT_REF_LOCI_13"...
INFO:root:...Done
INFO:root:Retrieving "alt.scaf.fna.gz" for group "ALT_REF_LOCI_14"...
INFO:root:...Done
INFO:root:Retrieving "alt.scaf.fna.gz" for group "ALT_REF_LOCI_15"...
INFO:root:...Done
INFO:root:Retrieving "alt.scaf.fna.gz" for group "ALT_REF_LOCI_16"...
INFO:root:...Done
INFO:root:Retrieving "alt.scaf.fna.gz" for group "ALT_REF_LOCI_17"...
INFO:root:...Done
INFO:root:Retrieving "alt.scaf.fna.gz" for group "ALT_REF_LOCI_18"...
INFO:root:...Done
INFO:root:Retrieving "alt.scaf.fna.gz" for group "ALT_REF_LOCI_19"...
INFO:root:...Done
INFO:root:Retrieving "alt.scaf.fn

ALT_REF_LOCI_1: 187
ALT_REF_LOCI_10: 1
ALT_REF_LOCI_11: 1
ALT_REF_LOCI_12: 1
ALT_REF_LOCI_13: 1
ALT_REF_LOCI_14: 1
ALT_REF_LOCI_15: 1
ALT_REF_LOCI_16: 1
ALT_REF_LOCI_17: 1
ALT_REF_LOCI_18: 1
ALT_REF_LOCI_19: 1
ALT_REF_LOCI_2: 26
ALT_REF_LOCI_20: 1
ALT_REF_LOCI_21: 1
ALT_REF_LOCI_22: 1
ALT_REF_LOCI_23: 1
ALT_REF_LOCI_24: 1
ALT_REF_LOCI_25: 1
ALT_REF_LOCI_26: 1
ALT_REF_LOCI_27: 1
ALT_REF_LOCI_28: 1
ALT_REF_LOCI_29: 1
ALT_REF_LOCI_3: 7
ALT_REF_LOCI_30: 1
ALT_REF_LOCI_31: 1
ALT_REF_LOCI_32: 1
ALT_REF_LOCI_33: 1
ALT_REF_LOCI_34: 1
ALT_REF_LOCI_35: 1
ALT_REF_LOCI_4: 3
ALT_REF_LOCI_5: 3
ALT_REF_LOCI_6: 3
ALT_REF_LOCI_7: 3
ALT_REF_LOCI_8: 2
ALT_REF_LOCI_9: 1
Combined len = 261
Tot of each individual = 261


### Get a FASTA sequence

In [25]:
ident = dst_counter.most_common(1)[0][0] # CM000681.2

def fasta_from_entrez(acc):
    url = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&amp;id=%s&amp;rettype=fasta&amp;retmode=text' % acc
    with urllib.request.urlopen(url) as response:
        return parse_fasta(response)

fasta = fasta_from_entrez(ident)
for k, v in fasta.items():
    print('%s, (%d): %s' % (k, len(v), v[:100]))

CM000681.2, (58617616): NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN


### Create a stacked alignment

In [26]:
chr19seq = fasta['CM000681.2']

In [27]:
aln = als['CM000681.2'][0]
aln

CM000681.2[20663140:21042381]->GL383573.1

In [28]:
altseq = combined['GL383573.1']
altseq[:1000]

'GTTTACCCTTAAAGGTATTGTGTGCGTGTCTTTTCTTCTCCCCTCATGTGTTTCCTGCACAGAACATATTTCTCAAAGGCTTCATTTGTTCCTTTTTATTCATTTTTGTTTAATCTTGTCTGCATGCCTTATTTTGGCAAGGTGGTCTTCAAACTCTGATGTTCTTTCTTCTGCTTGGTCAATGCAGCTACAAATACTTTTGTATGCTTCAGGAGGTTCTCATGCTGTGTGTTTTAGCTCCATCAGATCATTTATGTTCCTGTCTAAACCAGTTATTCTAGTTAGCAGTTCCTCTAACCTTTTATCAAGGTTCTTAGCTTCTTTCAATTGGTTTAGAACATGTTCCTTTAGCTCAGCAGAGTTTATTACCCATCTTCTGAAGCCTACTTTTGTCAATTCATCCATCTCATCCTCTGTCCAGTTCTGCACCCTTGCTGAAGAGACATTTTGATAACTTGGAGGAAAAGCCGCACTCTGGCCTTTCAGAGTTCTTTTGTTGATTCTTTCTCATCTTCATGAACTTGTCTAGTTTTGATCTTTGAGGCTGTTGACAACTGAATGGTTGTTTCTGTTTTGTTTTGTTTTGTTTTGAGACAGAGTCTCACTCTGTCACCCAGGCTGGAGGGCAGTGGCACAATCTCGGCTCAATGCAACCTTCGCATCCCAGGTTCAAGTGATTCTCGTGTCTCAGATTCCCGAGTAGCTGGGATTACAGGCATGCAACACAACGCCTGGCTAATTTTTGTATTTTTTAGTAGAGATGGGGTTTTGCCATGTTGGCCAGGCTAGTTTTGAACTCCTGACCTCAAGTGATCTGCCTGTCTCAGCCTCCCAAAGTACTGAGATTACTGGTGTGAGCCACCATGCCCGGCCTTGGATGGGGTTTTTGTCGGTACTTTTTTGTTGTTGATGTTGTTGTTTTCTGTTTGTTTGTTTTTCTTTCAGTGGTCAGGTCCCTCTTCTGTAGGGCTGCTGCAGTTTGCTGGGGGTTCACTTCAG

In [29]:
st, en = als['CM000681.2'][0].st, als['CM000681.2'][0].en
st, en

(20663140, 21042381)

In [30]:
# offsets were already adjusted to be 0 based when we
# constructed the Alignment object
chr19seq[st:st+30]

'GTTTACCCTTAAAGGTATTGTGTGCGTGTC'

In [31]:
altseq[:30]

'GTTTACCCTTAAAGGTATTGTGTGCGTGTC'

In [32]:
# do they seem to line up at the beginning?
chr19seq[st:st+30] == altseq[:30]
# ...yes

True

In [33]:
# Create stacked alignment out of sequences and CIGAR
def cigar_to_stacked(refseq, al):
    altid = al.target
    altseq = combined[altid]
    st, en = al.st, al.en
    refs, alts = [], []
    refsubstr = refseq[st:en]
    refoff, altoff = 0, 0
    last_op = None
    for op, run in al.cigar:
        assert last_op is None or last_op != op
        if op == 'M':
            refs.append(refsubstr[refoff:refoff+run])
            alts.append(altseq[altoff:altoff+run])
            refoff += run
            altoff += run
        elif op == 'D':
            refs.append(refsubstr[refoff:refoff+run])
            alts.append('-' * run)
            refoff += run
        elif op == 'I':
            refs.append('-' * run)
            alts.append(altseq[altoff:altoff+run])
            altoff += run
        last_op = op
    return ''.join(refs), ''.join(alts)

In [34]:
# For prettier printing of these large stacked alignments
def flow_stacked(refs, alts, per_line=80):
    off = 0
    lines = []
    while off < len(refs):
        refline = refs[off:off+per_line]
        altline = alts[off:off+per_line]
        bars = []
        for rc, ac in zip(refline, altline):
            bars.append('|' if rc == ac else ' ')
        lines.append(refline)
        lines.append(''.join(bars))
        lines.append(altline)
        lines.append('\n')
        off += per_line
    return lines

In [35]:
# Try for first ALT locus on chr19
lines = flow_stacked(*cigar_to_stacked(chr19seq, als['CM000681.2'][0]))
print('\n'.join(lines))

GTTTACCCTTAAAGGTATTGTGTGCGTGTCTTTTCTTCTCCCCTCATGTGTTTCCTGCACAGAACATATTTCTCAAAGGC
||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
GTTTACCCTTAAAGGTATTGTGTGCGTGTCTTTTCTTCTCCCCTCATGTGTTTCCTGCACAGAACATATTTCTCAAAGGC


TTCATTTGTTCCTTTTTATTCATTTTTGTTTAATCTTGTCTGCATGCCTTATTTTGGCAAGGTGGTCTTCAAACTCTGAT
||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
TTCATTTGTTCCTTTTTATTCATTTTTGTTTAATCTTGTCTGCATGCCTTATTTTGGCAAGGTGGTCTTCAAACTCTGAT


GTTCTTTCTTCTGCTTGGTCAATGCAGCTACAAATACTTTTGTATGCTTCAGGAGGTTCTCATGCTGTGTGTTTTAGCTC
||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
GTTCTTTCTTCTGCTTGGTCAATGCAGCTACAAATACTTTTGTATGCTTCAGGAGGTTCTCATGCTGTGTGTTTTAGCTC


CATCAGATCATTTATGTTCCTGTCTAAACCAGTTATTCTAGTTAGCAGTTCCTCTAACCTTTTATCAAGGTTCTTAGCTT
||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
CATCAGATCATTTATGTTCCTGTCTAAACCAGTTATTCTAGTTAGCAGTTCCTCTAACCTTTTATCAAGGTTCTTAGCTT


CTTTCAATTGGTTTAGAACA

In [36]:
# Try for second ALT locus on chr19
lines = flow_stacked(*cigar_to_stacked(chr19seq, als['CM000681.2'][1]))
print('\n'.join(lines))

CAGCTAACAAAGTAACACCAATATGGTACCCAGGTCGCATAAGCACTCAAGTGTATCCAGAAAGATGAGCAATTCTTTTT
||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
CAGCTAACAAAGTAACACCAATATGGTACCCAGGTCGCATAAGCACTCAAGTGTATCCAGAAAGATGAGCAATTCTTTTT


TTTTATATATACTTTAAGTTTTAGGGTACATGTGCACAACACACAGGTTAGTTACACATGTATACATGTGCCATGTTGGT
||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
TTTTATATATACTTTAAGTTTTAGGGTACATGTGCACAACACACAGGTTAGTTACACATGTATACATGTGCCATGTTGGT


GTGCTGCACCCATTGAGCAATTCTTTAAAATGTGATCCCATGAAATATTTCTAACTTTTCACAATCTCAAAAAGTAAGTG
||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
GTGCTGCACCCATTGAGCAATTCTTTAAAATGTGATCCCATGAAATATTTCTAACTTTTCACAATCTCAAAAAGTAAGTG


ACAGCTATTGCTCTAAATATCACTTTCAATAATATAGTACTTGTAGCAAGGCATCCTATAATTTTAAGAGCTACATAATG
||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
ACAGCTATTGCTCTAAATATCACTTTCAATAATATAGTACTTGTAGCAAGGCATCCTATAATTTTAAGAGCTACATAATG


TAAATTACGTAATTTACATA